# Level 3: Advanced Architecture Design - CIFAR-10 Classification

## Objective
Design custom architecture with attention mechanisms and multi-scale features.

## Target Accuracy: ≥91%

## Deliverables:
- ✅ Architecture design document
- ✅ Custom model implementation
- ✅ Per-class performance analysis
- ✅ Visualization/interpretability (Grad-CAM)
- ✅ Insights and findings

---

In [ ]:
# Install required packages
# !pip install torch torchvision matplotlib seaborn tqdm albumentations grad-cam timm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import os

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Architecture Design Document

### Custom Architecture: AttentionResNet

**Design Philosophy:**
- Use ResNet backbone with custom attention modules
- Add Squeeze-and-Excitation (SE) blocks for channel attention
- Implement Spatial Attention for focusing on important regions
- Multi-scale feature aggregation for better representation

**Architecture Components:**
1. **Backbone**: Modified ResNet50 with SE attention
2. **CBAM**: Convolutional Block Attention Module
3. **Multi-Scale Pooling**: Global Average + Max Pooling
4. **Custom Classifier**: Deep MLP with residual connections

In [ ]:
# CIFAR-10 setup
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
           'dog', 'frog', 'horse', 'ship', 'truck']
NUM_CLASSES = 10
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

## 2. Custom Attention Modules

In [ ]:
class SqueezeExcitation(nn.Module):
    """Squeeze-and-Excitation block for channel attention"""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)
        
    def forward(self, x):
        b, c, _, _ = x.size()
        # Squeeze: Global average pooling
        y = F.adaptive_avg_pool2d(x, 1).view(b, c)
        # Excitation: FC -> ReLU -> FC -> Sigmoid
        y = F.relu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)).view(b, c, 1, 1)
        return x * y


class ChannelAttention(nn.Module):
    """Channel attention module"""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        
    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        out = torch.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * out


class SpatialAttention(nn.Module):
    """Spatial attention module"""
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)
        out = torch.sigmoid(self.conv(concat))
        return x * out


class CBAM(nn.Module):
    """Convolutional Block Attention Module"""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention()
        
    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x

print("Attention modules defined: SE, Channel, Spatial, CBAM")

## 3. Custom AttentionResNet Architecture

In [ ]:
class AttentionResNet(nn.Module):
    """
    Custom ResNet with Attention Mechanisms
    
    Architecture:
    - ResNet50 backbone (pretrained)
    - CBAM attention after layer3 and layer4
    - Multi-scale pooling (avg + max)
    - Deep classifier with residual connection
    """
    def __init__(self, num_classes=10, dropout_rate=0.5):
        super().__init__()
        
        # Load pretrained ResNet50
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        
        # Extract backbone layers
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1  # 256 channels
        self.layer2 = resnet.layer2  # 512 channels
        self.layer3 = resnet.layer3  # 1024 channels
        self.layer4 = resnet.layer4  # 2048 channels
        
        # Attention modules
        self.cbam3 = CBAM(1024)
        self.cbam4 = CBAM(2048)
        
        # Multi-scale pooling
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.maxpool_final = nn.AdaptiveMaxPool2d(1)
        
        # Classifier with residual connection
        self.classifier = nn.Sequential(
            nn.Linear(2048 * 2, 1024),  # *2 for concat of avg and max pool
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.6),
            nn.Linear(512, num_classes)
        )
        
    def forward(self, x):
        # Backbone
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        
        # Apply CBAM after layer3
        x = self.layer3(x)
        x = self.cbam3(x)
        
        # Apply CBAM after layer4
        x = self.layer4(x)
        x = self.cbam4(x)
        
        # Multi-scale pooling
        avg_out = self.avgpool(x).flatten(1)
        max_out = self.maxpool_final(x).flatten(1)
        x = torch.cat([avg_out, max_out], dim=1)
        
        # Classifier
        x = self.classifier(x)
        return x

# Create and inspect model
model = AttentionResNet(NUM_CLASSES).to(device)
print(f"\nModel: AttentionResNet")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 4. Data Loading with Advanced Augmentation

In [ ]:
# Custom Dataset for Albumentations
class CIFAR10Album(Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        image = np.array(image)
        if self.transform:
            image = self.transform(image=image)['image']
        return image, label

# Advanced augmentation
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.5),
    A.CoarseDropout(max_holes=8, max_height=20, max_width=20, p=0.3),
    A.ColorJitter(0.2, 0.2, 0.2, 0.1, p=0.4),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
])

test_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
])

# Load data
raw_train = datasets.CIFAR10(root='./data', train=True, download=True)
raw_test = datasets.CIFAR10(root='./data', train=False, download=True)

# Split
indices = list(range(len(raw_train)))
np.random.shuffle(indices)
train_idx, val_idx = indices[:40000], indices[40000:]

train_dataset = CIFAR10Album(Subset(raw_train, train_idx), train_transform)
val_dataset = CIFAR10Album(Subset(raw_train, val_idx), test_transform)
test_dataset = CIFAR10Album(raw_test, test_transform)

# Loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

## 5. Training

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in tqdm(loader, desc='Training'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(loader), 100. * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return running_loss / len(loader), 100. * correct / total, np.array(all_preds), np.array(all_labels)

In [ ]:
# Training configuration
NUM_EPOCHS = 25
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

# Training loop
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_model_state = None

print("="*60)
print("Training AttentionResNet on CIFAR-10")
print("="*60)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    scheduler.step()
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = model.state_dict().copy()
        print(f"*** New best: {val_acc:.2f}% ***")

print(f"\nTraining Complete! Best Val Acc: {best_val_acc:.2f}%")

In [ ]:
# Save model
torch.save(best_model_state, 'models/attention_resnet_cifar10_level3.pth')

# Test evaluation
model.load_state_dict(best_model_state)
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)

print(f"\n{'='*60}")
print(f"TEST RESULTS - AttentionResNet")
print(f"{'='*60}")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Target: ≥91%")
print(f"Status: {'✅ PASSED' if test_acc >= 91 else '❌ NOT PASSED'}")
print(f"{'='*60}")

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=2)
axes[0].plot(epochs, history['val_loss'], 'r-', label='Val', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['train_acc'], 'b-', label='Train', linewidth=2)
axes[1].plot(epochs, history['val_acc'], 'r-', label='Val', linewidth=2)
axes[1].axhline(y=91, color='g', linestyle='--', label='Target (91%)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/training_curves_level3.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Per-Class Performance Analysis

In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix - Level 3 (Test Acc: {test_acc:.2f}%)')
plt.tight_layout()
plt.savefig('results/confusion_matrix_level3.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=CLASSES))

In [ ]:
# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100

plt.figure(figsize=(12, 6))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(CLASSES)))
bars = plt.bar(CLASSES, per_class_acc, color=colors, edgecolor='black')

for bar, acc in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=10)

plt.axhline(y=91, color='r', linestyle='--', label='Target (91%)')
plt.xlabel('Class')
plt.ylabel('Accuracy (%)')
plt.title('Per-Class Accuracy - AttentionResNet')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig('results/per_class_accuracy_level3.png', dpi=150, bbox_inches='tight')
plt.show()

# Analysis
print("\nPer-Class Analysis:")
print("-"*50)
sorted_idx = np.argsort(per_class_acc)
print("\nHardest classes (lowest accuracy):")
for i in sorted_idx[:3]:
    print(f"  {CLASSES[i]}: {per_class_acc[i]:.2f}%")
print("\nEasiest classes (highest accuracy):")
for i in sorted_idx[-3:]:
    print(f"  {CLASSES[i]}: {per_class_acc[i]:.2f}%")

## 8. Grad-CAM Visualization

In [ ]:
class GradCAM:
    """Simple Grad-CAM implementation"""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def __call__(self, x, class_idx=None):
        self.model.eval()
        output = self.model(x)
        
        if class_idx is None:
            class_idx = output.argmax(dim=1)
        
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, class_idx] = 1
        output.backward(gradient=one_hot)
        
        # Compute weights
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        
        return cam.squeeze().cpu().numpy()

print("Grad-CAM class defined")

In [ ]:
def visualize_gradcam(model, test_dataset, num_samples=8):
    """Visualize Grad-CAM for sample images"""
    model.eval()
    gradcam = GradCAM(model, model.layer4)
    
    fig, axes = plt.subplots(2, num_samples, figsize=(20, 6))
    indices = np.random.choice(len(test_dataset), num_samples, replace=False)
    
    for i, idx in enumerate(indices):
        img, label = test_dataset[idx]
        img_tensor = img.unsqueeze(0).to(device)
        
        # Get prediction
        with torch.no_grad():
            pred = model(img_tensor).argmax(dim=1).item()
        
        # Get Grad-CAM
        cam = gradcam(img_tensor, pred)
        
        # Denormalize image
        img_np = img.permute(1, 2, 0).numpy()
        img_np = img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        img_np = np.clip(img_np, 0, 1)
        
        # Original image
        axes[0, i].imshow(img_np)
        axes[0, i].set_title(f'True: {CLASSES[label]}', fontsize=10)
        axes[0, i].axis('off')
        
        # Grad-CAM overlay
        axes[1, i].imshow(img_np)
        axes[1, i].imshow(cam, cmap='jet', alpha=0.5)
        axes[1, i].set_title(f'Pred: {CLASSES[pred]}', fontsize=10)
        axes[1, i].axis('off')
    
    axes[0, 0].set_ylabel('Original', fontsize=12)
    axes[1, 0].set_ylabel('Grad-CAM', fontsize=12)
    
    plt.suptitle('Grad-CAM Visualization - Model Attention', fontsize=14)
    plt.tight_layout()
    plt.savefig('results/gradcam_level3.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_gradcam(model, test_dataset)

## 9. Insights and Findings

### Architecture Insights:
1. **CBAM Attention**: Channel and spatial attention help focus on discriminative features
2. **Multi-scale Pooling**: Combining avg and max pooling captures both average and salient features
3. **Deep Classifier**: Multiple FC layers with BatchNorm improve generalization

### Performance Analysis:
- Similar classes (cat/dog, airplane/ship) show confusion
- Attention helps model focus on key discriminative regions
- Grad-CAM shows model attends to semantically meaningful parts

### Recommendations for Improvement:
- Use ensemble of multiple attention models
- Try Vision Transformers for global attention
- Implement mixup/cutmix augmentation

In [ ]:
# Final Summary
print("="*60)
print("LEVEL 3 SUMMARY - ADVANCED ARCHITECTURE")
print("="*60)
print(f"\nArchitecture: AttentionResNet (ResNet50 + CBAM)")
print(f"Key Features:")
print(f"  - Channel Attention (SE-like)")
print(f"  - Spatial Attention")
print(f"  - Multi-scale Pooling")
print(f"\nTest Accuracy: {test_acc:.2f}%")
print(f"Target: ≥91%")
print(f"Status: {'✅ PASSED' if test_acc >= 91 else '❌ NOT PASSED'}")
print("="*60)